# 01 Data Cleaning — Google Maps Restaurant Dataset

This notebook cleans semi-structured Google Maps restaurant data for market intelligence analysis.

Cleaning logic:
1. Inspect raw data before transformation
2. Create a stable `clean` dataframe from selected columns
3. Standardize missing values
4. Convert numeric columns before metric calculations
5. Engineer geographic, price, service, digital, and rating features
6. Validate data quality: missing values, star totals, duplicates
7. Export cleaned dataset for Power BI

In [3]:
import json
import pandas as pd
import numpy as np
import re
from pathlib import Path

path = "../data/raw/gmaps_db.restaurants1.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

raw = pd.json_normalize(data)

raw.shape

(1625, 34)

In [4]:
raw.head()

,_id,Address,Avg_Rating,Category,Latitude,Longitude,Name,Opening_Hours,Phone,Rating_1_Stars,...,Highlights,Menu_Link,Opening_Hours.$numberDouble,Phone.$numberDouble,Service_Options.$numberDouble,Avg_Rating.$numberDouble,Total_Reviews.$numberDouble,Category.$numberDouble,Address.$numberDouble,Price_Range
0,https://www.google.com/maps/place/Nh%C3%A0+H%C...,"Đường Bờ Sông, Lái Thiêu, Thuận An, Bình Dương...",3.9,Mở cả ngày,10.896955,106.695794,Nhà Hàng Ẩm Thực Bờ Sông,Thứ Hai Mở cửa cả ngày | Thứ Ba Mở cửa cả ngày...,+84 784 054 898,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.google.com/maps/place/Subin+BBQ+Vi...,"79 ĐT743B, Khu Phố Thống Nhất 1, Thành Phố, Bì...",4.6,Thịt nướng,10.908412,106.741475,Subin BBQ Vincom Dĩ An Buffet Nướng Hàn Quốc,Thứ Hai 10:30–22:00 | Thứ Ba 10:30–22:00 | Thứ...,+84 274 6268 608,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.google.com/maps/place/L%C3%A0ng+%E...,"1/6 ĐT743B khu phố Bình Đức 1, Thuận An, Bình ...",4.1,Nhà hàng,10.901151,106.713094,Làng Ẩm Thực 126 - Tiệc trọn gói hoàn hảo tại ...,Thứ Hai 09:00–23:30 | Thứ Ba 09:00–23:30 | Thứ...,+84 912 715 126,6,...,Chào đón những người thuộc giới LGBTQ+,https://langamthuc126.com/,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.google.com/maps/place/Nh%C3%A0+h%C...,"16 Võ Thị Sáu, Đông Hòa, Dĩ An, Bình Dương, Vi...",3.7,Mở cả ngày,10.891235,106.774655,Nhà hàng ẨM THỰC SÂN VƯỜN,NaN,+84 902 913 622,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.google.com/maps/place/%E1%BA%A8m+T...,"637/10/49 Tổ 30, Khu Phố 2, Quận 12, Hồ Chí Mi...",4.3,Dành cho gia đình,10.888727,106.675828,Ẩm Thực Song Phụng 2 I Quán Ăn Sân Vườn Quận 1...,Thứ Hai 10:00–22:00 | Thứ Ba 10:00–22:00 | Thứ...,+84 965 668 839,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Initial schema and missing-value audit

In [5]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1625 entries, 0 to 1624
Data columns (total 34 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   _id                            1625 non-null   object 
 1   Address                        1624 non-null   object 
 2   Avg_Rating                     1622 non-null   float64
 3   Category                       1624 non-null   object 
 4   Latitude                       1625 non-null   float64
 5   Longitude                      1625 non-null   float64
 6   Name                           1625 non-null   object 
 7   Opening_Hours                  1437 non-null   object 
 8   Phone                          1547 non-null   object 
 9   Rating_1_Stars                 1625 non-null   object 
 10  Rating_2_Stars                 1625 non-null   object 
 11  Rating_3_Stars                 1625 non-null   object 
 12  Rating_4_Stars                 1625 non-null   o

In [6]:
raw[[
    "Name", "Address", "Category",
    "Avg_Rating", "Total_Reviews",
    "Average_Price", "Price_Range.$numberDouble",
    "Service_Options", "Website", "Menu_Link"
]].head(10)

,Name,Address,Category,Avg_Rating,Total_Reviews,Average_Price,Price_Range.$numberDouble,Service_Options,Website,Menu_Link
0,Nhà Hàng Ẩm Thực Bờ Sông,"Đường Bờ Sông, Lái Thiêu, Thuận An, Bình Dương...",Mở cả ngày,3.9,265.0,NaN,NaN,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng gián tiếp",NaN,NaN
1,Subin BBQ Vincom Dĩ An Buffet Nướng Hàn Quốc,"79 ĐT743B, Khu Phố Thống Nhất 1, Thành Phố, Bì...",Thịt nướng,4.6,461.0,200.000-300.000 ₫/người,NaN,Ăn tại chỗ,facebook.com,NaN
2,Làng Ẩm Thực 126 - Tiệc trọn gói hoàn hảo tại ...,"1/6 ĐT743B khu phố Bình Đức 1, Thuận An, Bình ...",Nhà hàng,4.1,61.0,NaN,NaN,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng gián tiếp",langamthuc126.com,https://langamthuc126.com/
3,Nhà hàng ẨM THỰC SÂN VƯỜN,"16 Võ Thị Sáu, Đông Hòa, Dĩ An, Bình Dương, Vi...",Mở cả ngày,3.7,18.0,300.000-400.000 ₫/người,NaN,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng",NaN,NaN
4,Ẩm Thực Song Phụng 2 I Quán Ăn Sân Vườn Quận 1...,"637/10/49 Tổ 30, Khu Phố 2, Quận 12, Hồ Chí Mi...",Dành cho gia đình,4.3,47.0,NaN,NaN,"Ăn tại chỗ, Mua hàng ngay trên xe, Giao hàng g...",amthucsongphung2.com,NaN
5,ẨM THỰC SÔNG XANH,"35 Khu phố Long thới Phường Lái Thiêu, Tp, Thu...",· ,4.1,1094.0,NaN,NaN,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng",amthucsongxanh.com,NaN
6,Nhà Hàng Dìn Ký Phú Long,"Số 31/4, Đường Đê bao sông Sài Gòn, Khu ph...",Mở cả ngày,4.3,327.0,NaN,NaN,"Ăn tại chỗ, Nhận hàng ở lề đường, Giao hàng gi...",NaN,NaN
7,Nhà Hàng Vạn Lộc Phát,"68-68A/12 QL13, Khu phố Trung, Thuận An, Bình ...",Món ăn Việt Nam,4.5,889.0,NaN,NaN,"Ăn tại chỗ, Mua hàng ngay trên xe, Giao hàng g...",nhahangvanlocphat.vn,https://nhahangvanlocphat.vn/thuc-don
8,Năm Lửa 6,"1 Đường N2, Khu Phố Thống Nhất 1, Dĩ An, Bình ...",Nhà hàng,3.9,1008.0,NaN,NaN,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng",NaN,NaN
9,Béo BBQ,"45 Nguyễn Du, Dĩ An, Bình Dương, Việt Nam",· ,4.4,713.0,NaN,NaN,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng gián tiếp",NaN,http://beo-quan.business.site/


Initial observation:
- The dataset contains restaurant-level Google Maps data.
- Some fields are already usable, such as Name, Address, Avg_Rating, Total_Reviews, Latitude, Longitude.
- Some fields are semi-structured or unreliable, especially Average_Price, Price_Range, Website, Menu_Link, and Service_Options.

In [7]:
schema = pd.DataFrame({
    "column": raw.columns,
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum().values,
    "missing_rate": (raw.isna().mean().values * 100).round(2)
})

schema.sort_values("missing_rate", ascending=False).head(20)

,column,dtype,missing_count,missing_rate
Address.$numberDouble,Address.$numberDouble,object,1624,99.94
Avg_Rating.$numberDouble,Avg_Rating.$numberDouble,object,1624,99.94
Category.$numberDouble,Category.$numberDouble,object,1624,99.94
Total_Reviews.$numberDouble,Total_Reviews.$numberDouble,object,1624,99.94
Service_Options.$numberDouble,Service_Options.$numberDouble,object,1590,97.85
Phone.$numberDouble,Phone.$numberDouble,object,1547,95.20
Opening_Hours.$numberDouble,Opening_Hours.$numberDouble,object,1437,88.43
Website.$numberDouble,Website.$numberDouble,object,1354,83.32
Average_Price.$numberDouble,Average_Price.$numberDouble,object,1313,80.80
Menu_Link.$numberDouble,Menu_Link.$numberDouble,object,1121,68.98


In [8]:
#Missing values
missing_summary = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_rate_%": (raw.isna().mean() * 100).round(2)
}).sort_values("missing_rate_%", ascending=False)

missing_summary.head(25)

,missing_count,missing_rate_%
Address.$numberDouble,1624,99.94
Avg_Rating.$numberDouble,1624,99.94
Category.$numberDouble,1624,99.94
Total_Reviews.$numberDouble,1624,99.94
Service_Options.$numberDouble,1590,97.85
Phone.$numberDouble,1547,95.20
Opening_Hours.$numberDouble,1437,88.43
Website.$numberDouble,1354,83.32
Average_Price.$numberDouble,1313,80.80
Menu_Link.$numberDouble,1121,68.98


Average_Price, Website, Menu_Link, and Opening_Hours have high missing rates. Therefore, they should not be used as direct ranking metrics. Instead, they can be transformed into business indicators such as price availability and digital readiness.


## 2. Create the clean dataframe

Important: the raw JSON has some fields split into normal columns and `$numberDouble` columns.  
This cell creates a stable `clean` dataframe before any feature engineering.

In [9]:
cols = [
    "_id", "Name", "Address", "Category",
    "Avg_Rating", "Total_Reviews",
    "Rating_1_Stars", "Rating_2_Stars", "Rating_3_Stars",
    "Rating_4_Stars", "Rating_5_Stars",
    "Latitude", "Longitude",
    "Average_Price", "Opening_Hours",
    "Service_Options", "Phone", "Website", "Menu_Link",
    "Highlights", "URL", "Price_Range"
]

clean = pd.DataFrame()

for c in cols:
    normal_col = c
    nested_col = c + ".$numberDouble"

    if normal_col in raw.columns:
        clean[c] = raw[normal_col]
    else:
        clean[c] = np.nan

    if nested_col in raw.columns:
        clean[c] = clean[c].where(clean[c].notna(), raw[nested_col])

clean.head()

,_id,Name,Address,Category,Avg_Rating,Total_Reviews,Rating_1_Stars,Rating_2_Stars,Rating_3_Stars,Rating_4_Stars,...,Longitude,Average_Price,Opening_Hours,Service_Options,Phone,Website,Menu_Link,Highlights,URL,Price_Range
0,https://www.google.com/maps/place/Nh%C3%A0+H%C...,Nhà Hàng Ẩm Thực Bờ Sông,"Đường Bờ Sông, Lái Thiêu, Thuận An, Bình Dương...",Mở cả ngày,3.9,265.0,34,7,34,75,...,106.695794,NaN,Thứ Hai Mở cửa cả ngày | Thứ Ba Mở cửa cả ngày...,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng gián tiếp",+84 784 054 898,NaN,NaN,NaN,https://www.google.com/maps/place/Nh%C3%A0+H%C...,NaN
1,https://www.google.com/maps/place/Subin+BBQ+Vi...,Subin BBQ Vincom Dĩ An Buffet Nướng Hàn Quốc,"79 ĐT743B, Khu Phố Thống Nhất 1, Thành Phố, Bì...",Thịt nướng,4.6,461.0,26,9,17,32,...,106.741475,200.000-300.000 ₫/người,Thứ Hai 10:30–22:00 | Thứ Ba 10:30–22:00 | Thứ...,Ăn tại chỗ,+84 274 6268 608,facebook.com,NaN,NaN,https://www.google.com/maps/place/Subin+BBQ+Vi...,NaN
2,https://www.google.com/maps/place/L%C3%A0ng+%E...,Làng Ẩm Thực 126 - Tiệc trọn gói hoàn hảo tại ...,"1/6 ĐT743B khu phố Bình Đức 1, Thuận An, Bình ...",Nhà hàng,4.1,61.0,6,2,5,12,...,106.713094,NaN,Thứ Hai 09:00–23:30 | Thứ Ba 09:00–23:30 | Thứ...,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng gián tiếp",+84 912 715 126,langamthuc126.com,https://langamthuc126.com/,Chào đón những người thuộc giới LGBTQ+,https://www.google.com/maps/place/L%C3%A0ng+%E...,NaN
3,https://www.google.com/maps/place/Nh%C3%A0+h%C...,Nhà hàng ẨM THỰC SÂN VƯỜN,"16 Võ Thị Sáu, Đông Hòa, Dĩ An, Bình Dương, Vi...",Mở cả ngày,3.7,18.0,5,0,1,1,...,106.774655,300.000-400.000 ₫/người,NaN,"Ăn tại chỗ, Đồ ăn mang đi, Giao hàng",+84 902 913 622,NaN,NaN,NaN,https://www.google.com/maps/place/Nh%C3%A0+h%C...,NaN
4,https://www.google.com/maps/place/%E1%BA%A8m+T...,Ẩm Thực Song Phụng 2 I Quán Ăn Sân Vườn Quận 1...,"637/10/49 Tổ 30, Khu Phố 2, Quận 12, Hồ Chí Mi...",Dành cho gia đình,4.3,47.0,3,3,3,4,...,106.675828,NaN,Thứ Hai 10:00–22:00 | Thứ Ba 10:00–22:00 | Thứ...,"Ăn tại chỗ, Mua hàng ngay trên xe, Giao hàng g...",+84 965 668 839,amthucsongphung2.com,NaN,NaN,https://www.google.com/maps/place/%E1%BA%A8m+T...,NaN


## 3. Standardize missing values



In [10]:
clean = clean.replace({
    "NaN": np.nan,
    "nan": np.nan,
    "N/A": np.nan,
    "": np.nan,
    "None": np.nan
})

clean = clean.infer_objects(copy=False)

clean[["Name", "Address", "Average_Price", "Service_Options", "Website", "Menu_Link"]].isna().mean() * 100

C:\Users\lihoang14\AppData\Local\Temp\ipykernel_26548\813133155.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clean = clean.replace({


Name                0.000000
Address             3.569231
Average_Price      49.661538
Service_Options     6.461538
Website            55.200000
Menu_Link          80.861538
dtype: float64

In [11]:
text_cols = [
    "Name", "Address", "Category", "Average_Price",
    "Opening_Hours", "Service_Options", "Phone",
    "Website", "Menu_Link", "Highlights", "URL"
]

for col in text_cols:
    clean[col] = (
        clean[col]
        .astype("string")
        .str.replace("\n", " | ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace("", "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

clean = clean.replace({
    "<NA>": np.nan,
    "nan": np.nan,
    "NaN": np.nan,
    "N/A": np.nan,
    "": np.nan
})

## 4. Convert numeric columns

This prevents errors such as:
`TypeError: unsupported operand type(s) for -: 'str' and 'float'`

Star-rating columns must be numeric before calculating `Star_Total` and `Review_Count_Diff`.

In [12]:
num_cols = [
    "Avg_Rating", "Total_Reviews",
    "Rating_1_Stars", "Rating_2_Stars", "Rating_3_Stars",
    "Rating_4_Stars", "Rating_5_Stars",
    "Latitude", "Longitude"
]

for c in num_cols:
    clean[c] = pd.to_numeric(clean[c], errors="coerce")

clean[num_cols].dtypes

Avg_Rating        float64
Total_Reviews     float64
Rating_1_Stars      int64
Rating_2_Stars      int64
Rating_3_Stars      int64
Rating_4_Stars      int64
Rating_5_Stars      int64
Latitude          float64
Longitude         float64
dtype: object

In [13]:
clean[num_cols].describe()

,Avg_Rating,Total_Reviews,Rating_1_Stars,Rating_2_Stars,Rating_3_Stars,Rating_4_Stars,Rating_5_Stars,Latitude,Longitude
count,1622.000000,1622.000000,1625.000000,1625.000000,1625.000000,1625.000000,1625.000000,1625.000000,1625.000000
mean,4.419914,605.434649,30.536615,14.781538,50.558154,115.308308,438.088615,10.790900,106.705701
std,0.365852,1281.619843,92.003661,46.369324,166.214931,359.681537,1203.688787,0.064853,0.065047
min,1.800000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10.614062,106.565773
25%,4.100000,31.000000,0.000000,0.000000,0.000000,1.000000,15.000000,10.756291,106.670764
50%,4.400000,196.500000,8.000000,3.000000,9.000000,24.000000,104.000000,10.786440,106.696545
75%,4.700000,669.250000,34.000000,15.000000,45.000000,101.000000,381.000000,10.836184,106.738230
max,5.000000,22395.000000,1328.000000,650.000000,2315.000000,4942.000000,18184.000000,11.104896,106.921454


## 5. Geographic feature engineering

Create two geography levels:
- `Province_City`: high-level market filter
- `Area`: district/city-level drill-down when the address is reliable



In [14]:
def extract_province_city(row):
    address = "" if pd.isna(row["Address"]) else str(row["Address"]).lower()
    name = "" if pd.isna(row["Name"]) else str(row["Name"]).lower()
    text = address + " " + name

    # HCM
    if (
        "hồ chí minh" in text
        or "ho chi minh" in text
        or "tp.hcm" in text
        or "hcm" in text
        or "sài gòn" in text
        or "sai gon" in text
        or "cần giờ" in text
        or "can gio" in text
        or "nhà bè" in text
        or "nha be" in text
    ):
        return "HCM"

    # Bình Dương
    if "bình dương" in text or "binh duong" in text:
        return "Bình Dương"

    # Đồng Nai
    if (
        "đồng nai" in text
        or "dong nai" in text
        or "nhơn trạch" in text
        or "nhon trach" in text
    ):
        return "Đồng Nai"

    # Long An
    if (
        "long an" in text
        or "cần giuộc" in text
        or "can giuoc" in text
    ):
        return "Long An"

    # Tây Ninh
    if (
        "tây ninh" in text
        or "tay ninh" in text
        or "phước lý" in text
        or "phuoc ly" in text
    ):
        return "Tây Ninh"

    return "Không rõ"


clean["Province_City"] = clean.apply(extract_province_city, axis=1)

In [15]:
import re

def extract_area(row):
    address = "" if pd.isna(row["Address"]) else str(row["Address"]).lower()
    name = "" if pd.isna(row["Name"]) else str(row["Name"]).lower()
    text = address + " " + name

    patterns = [
        # HCM
        ("Thủ Đức", r"\bthủ\s*đức\b|\bthu\s*duc\b|\bquận\s*0?2\b|\bquan\s*0?2\b|\bq\.?\s*0?2\b|\bquận\s*0?9\b|\bquan\s*0?9\b|\bq\.?\s*0?9\b"),
        ("Quận 12", r"\bquận\s*0?12\b|\bquan\s*0?12\b|\bq\.?\s*0?12\b"),
        ("Quận 11", r"\bquận\s*0?11\b|\bquan\s*0?11\b|\bq\.?\s*0?11\b"),
        ("Quận 10", r"\bquận\s*0?10\b|\bquan\s*0?10\b|\bq\.?\s*0?10\b"),
        ("Quận 8",  r"\bquận\s*0?8\b|\bquan\s*0?8\b|\bq\.?\s*0?8\b"),
        ("Quận 7",  r"\bquận\s*0?7\b|\bquan\s*0?7\b|\bq\.?\s*0?7\b"),
        ("Quận 6",  r"\bquận\s*0?6\b|\bquan\s*0?6\b|\bq\.?\s*0?6\b"),
        ("Quận 5",  r"\bquận\s*0?5\b|\bquan\s*0?5\b|\bq\.?\s*0?5\b"),
        ("Quận 4",  r"\bquận\s*0?4\b|\bquan\s*0?4\b|\bq\.?\s*0?4\b"),
        ("Quận 3",  r"\bquận\s*0?3\b|\bquan\s*0?3\b|\bq\.?\s*0?3\b"),
        ("Quận 1",  r"\bquận\s*0?1\b|\bquan\s*0?1\b|\bq\.?\s*0?1\b"),

        ("Bình Thạnh", r"\bbình\s*thạnh\b|\bbinh\s*thanh\b"),
        ("Gò Vấp", r"\bgò\s*vấp\b|\bgo\s*vap\b"),
        ("Tân Bình", r"\btân\s*bình\b|\btan\s*binh\b"),
        ("Tân Phú", r"\btân\s*phú\b|\btan\s*phu\b"),
        ("Phú Nhuận", r"\bphú\s*nhuận\b|\bphu\s*nhuan\b"),
        ("Bình Tân", r"\bbình\s*tân\b|\bbinh\s*tan\b"),
        ("Hóc Môn", r"\bhóc\s*môn\b|\bhoc\s*mon\b"),
        ("Bình Chánh", r"\bbình\s*chánh\b|\bbinh\s*chanh\b"),
        ("Nhà Bè", r"\bnhà\s*bè\b|\bnha\s*be\b"),
        ("Củ Chi", r"\bcủ\s*chi\b|\bcu\s*chi\b"),
        ("Cần Giờ", r"\bcần\s*giờ\b|\bcan\s*gio\b"),

        # Bình Dương
        ("Dĩ An", r"\bdĩ\s*an\b|\bdi\s*an\b"),
        ("Thuận An", r"\bthuận\s*an\b|\bthuan\s*an\b"),
        ("Thủ Dầu Một", r"\bthủ\s*dầu\s*một\b|\bthu\s*dau\s*mot\b"),
        ("Bến Cát", r"\bbến\s*cát\b|\bben\s*cat\b"),
        ("Tân Uyên", r"\btân\s*uyên\b|\btan\s*uyen\b"),
        ("Bàu Bàng", r"\bbàu\s*bàng\b|\bbau\s*bang\b"),

        # Đồng Nai
        ("Nhơn Trạch", r"\bnhơn\s*trạch\b|\bnhon\s*trach\b"),
        ("Biên Hòa", r"\bbiên\s*hòa\b|\bbien\s*hoa\b"),

        # Long An
        ("Cần Giuộc", r"\bcần\s*giuộc\b|\bcan\s*giuoc\b"),

        # Tây Ninh
        ("Tây Ninh khác", r"\btây\s*ninh\b|\btay\s*ninh\b|\bphước\s*lý\b|\bphuoc\s*ly\b"),
    ]

    for area, pattern in patterns:
        if re.search(pattern, text):
            return area

    province = row.get("Province_City", "Không rõ")

    if province == "HCM":
        return "HCM khác"
    elif province == "Bình Dương":
        return "Bình Dương khác"
    elif province == "Đồng Nai":
        return "Đồng Nai khác"
    elif province == "Long An":
        return "Long An khác"
    elif province == "Tây Ninh":
        return "Tây Ninh khác"
    else:
        return "Không rõ"


clean["Area"] = clean.apply(extract_area, axis=1)
clean["Area"].value_counts(dropna=False).head(20)

Area
Quận 1        323
Thủ Đức       225
Quận 3        104
Dĩ An          81
Bình Chánh     76
Quận 4         64
Nhơn Trạch     62
Quận 5         58
Quận 12        55
Không rõ       54
Quận 7         53
Nhà Bè         51
Bình Tân       47
Gò Vấp         46
Hóc Môn        44
Quận 6         35
Quận 11        34
Phú Nhuận      31
Tân Bình       31
Thuận An       27
Name: count, dtype: int64

In [16]:
hcm_other = clean[clean["Area"] == "Không rõ"][
    ["Name", "Address", "Area", "Province_City"]
]

hcm_other.head(50)

,Name,Address,Area,Province_City
640,Phường (Phương Huyền),<NA>,Không rõ,Không rõ
864,Hà Nội Xưa - QUÁN ĂN NGON GIA ĐÌNH HƯƠNG VỊ BẮC,<NA>,Không rõ,Không rõ
876,Quán Ăn Gia Đình PHƯƠNG NAM,<NA>,Không rõ,Không rõ
916,Bún thịt nướng Cô Hà,<NA>,Không rõ,Không rõ
924,Cô Lộc Đà Lạt Quán,<NA>,Không rõ,Không rõ
943,Nhà Hàng Quỳnh Anh,<NA>,Không rõ,Không rõ
951,Công viên Quán,<NA>,Không rõ,Không rõ
963,"Quán Ăn A Hùng ( Các Món Ăn Người Hoa Chiên, X...",<NA>,Không rõ,Không rõ
981,Quán Ăn Gia Đình Bảy Xị,<NA>,Không rõ,Không rõ
993,Phở Bảy Cù,<NA>,Không rõ,Không rõ


In [17]:
clean["Market_Scope"] = np.where(
    clean["Province_City"].isin(["HCM", "Bình Dương"]),
    "Main Market",
    "Outside Main Market"
)

## 6. Price audit and transformation

Price data is incomplete, so it should be used for segmentation, not as a mandatory ranking metric.

In [18]:
price_missing_by_area = (
    clean.assign(price_missing=clean["Average_Price"].isna())
    .groupby("Area")
    .agg(
        restaurants=("Name", "count"),
        missing_price_count=("price_missing", "sum"),
        missing_price_rate=("price_missing", "mean")
    )
    .sort_values("restaurants", ascending=False)
)

price_missing_by_area["available_price_count"] = (
    price_missing_by_area["restaurants"] - price_missing_by_area["missing_price_count"]
)

price_missing_by_area["missing_price_rate_%"] = (
    price_missing_by_area["missing_price_rate"] * 100
).round(2)

price_missing_by_area[
    ["restaurants", "available_price_count", "missing_price_count", "missing_price_rate_%"]
].head(20)

,restaurants,available_price_count,missing_price_count,missing_price_rate_%
Area,,,,
Quận 1,323,225,98,30.34
Thủ Đức,225,108,117,52.00
Quận 3,104,72,32,30.77
Dĩ An,81,31,50,61.73
Bình Chánh,76,27,49,64.47
Quận 4,64,45,19,29.69
Nhơn Trạch,62,19,43,69.35
Quận 5,58,33,25,43.10
Quận 12,55,25,30,54.55


Decision:
- Price_Range is not usable because it is mostly missing / empty.
- Average_Price is more informative because it contains text ranges such as "100.000-200.000 ₫/người".


In [19]:
def parse_price_tier(x):
    if pd.isna(x):
        return np.nan

    s = str(x)
    nums = re.findall(r"\d{1,3}(?:\.\d{3})*", s)

    if len(nums) == 0:
        return np.nan

    nums = [int(n.replace(".", "")) for n in nums]
    max_price = max(nums)

    if max_price <= 100000:
        return "<=100k"
    elif max_price <= 200000:
        return "100-200k"
    elif max_price <= 300000:
        return "200-300k"
    elif max_price <= 400000:
        return "300-400k"
    else:
        return "400k+"


clean["Price_Tier"] = clean["Average_Price"].apply(parse_price_tier)

clean[["Average_Price", "Price_Tier"]].head(20)

,Average_Price,Price_Tier
0,<NA>,NaN
1,200.000-300.000 ₫/người,200-300k
2,<NA>,NaN
3,300.000-400.000 ₫/người,300-400k
4,<NA>,NaN
5,<NA>,NaN
6,<NA>,NaN
7,<NA>,NaN
8,<NA>,NaN
9,<NA>,NaN


Because nearly half of restaurants do not provide price information, price should be used as a segmentation feature, not as a mandatory ranking metric.

In [20]:
clean["Price_Tier"].value_counts(dropna=False)

Price_Tier
NaN         807
<=100k      400
100-200k    240
200-300k    124
400k+        35
300-400k     19
Name: count, dtype: int64

## 7. Category quality audit

The raw `Category` field is noisy and sometimes contains opening status instead of restaurant type.

In [21]:
clean["Category"].value_counts(dropna=False).head(30)

Category
Đang mở cửa · Đóng cửa vào 22:00    186
Đang mở cửa · Đóng cửa vào 21:00    127
Đang mở cửa · Đóng cửa vào 23:00    117
Mở cả ngày                          111
Nhà hàng                             96
Đã đóng cửa · Mở cửa lúc 10:00       81
Đã đóng cửa · Mở cửa lúc 11:00       66
Đã đóng cửa · Mở cửa lúc 7:00        51
Đang mở cửa · Đóng cửa vào 22:30     48
Đã đóng cửa · Mở cửa lúc 9:00        47
Đã đóng cửa · Mở cửa lúc 8:00        43
·                                   36
Đã đóng cửa · Mở cửa lúc 16:00       36
Đã đóng cửa · Mở cửa lúc 6:00        26
Đã đóng cửa · Mở cửa lúc 10:30       26
Đang mở cửa · Đóng cửa vào 0:00      26
Đang mở cửa · Đóng cửa vào 23:30     26
Đang mở cửa · Đóng cửa vào 21:30     24
Đang mở cửa · Đóng cửa vào 20:00     23
Đã đóng cửa · Mở cửa lúc 7:30        21
Đã đóng cửa · Mở cửa lúc 15:00       19
Đã đóng cửa · Mở cửa lúc 6:30        15
Đang mở cửa · Đóng cửa vào 20:30     13
Đang mở cửa · Đóng cửa vào 17:00     12
Đã đóng cửa · Mở cửa lúc 16:30 

## 8. Service-option feature engineering

Category is noisy because many values describe opening status instead of cuisine or restaurant type. Therefore, I will not use raw Category as a primary business dimension.

- Keep Category only for reference.
- Do not use it for main charts.
- Use Price_Tier, Area, Service_Options, and Digital_Readiness instead because they are more reliable.

In [22]:
clean["Service_Options"].dropna().sample(20, random_state=42)

479                  Ăn tại chỗ, Đồ ăn mang đi, Giao hàng
1400               | Nhận hàng ở lề đường,  | Giao hàng
1575      | Ăn tại chỗ,  | Đồ ăn mang đi,  | Giao hàng
780     Ăn tại chỗ, Mua hàng ngay trên xe, Giao hàng g...
250                  Ăn tại chỗ, Đồ ăn mang đi, Giao hàng
446     Ăn tại chỗ, Mua hàng ngay trên xe, Giao hàng g...
987      | Ăn tại chỗ,  | Mua hàng ngay trên xe,  |...
1594                     | Ăn tại chỗ,  | Đồ ăn mang đi
373        Ăn tại chỗ, Đồ ăn mang đi, Giao hàng gián tiếp
443                             Ăn tại chỗ, Đồ ăn mang đi
51                   Ăn tại chỗ, Đồ ăn mang đi, Giao hàng
680                  Ăn tại chỗ, Đồ ăn mang đi, Giao hàng
498     Ăn tại chỗ, Mua hàng ngay trên xe, Giao hàng g...
1506     | Ăn tại chỗ,  | Nhận hàng ở lề đường,  | ...
1313      | Ăn tại chỗ,  | Đồ ăn mang đi,  | Giao hàng
77         Ăn tại chỗ, Đồ ăn mang đi, Giao hàng gián tiếp
848                      Ăn tại chỗ, Nhận hàng ở lề đường
1345          

In [23]:
service = clean["Service_Options"].fillna("").str.lower()

clean["Has_Dine_In"] = service.str.contains("ăn tại chỗ", regex=False).astype(int)
clean["Has_Takeaway"] = service.str.contains("đồ ăn mang đi", regex=False).astype(int)
clean["Has_Delivery"] = service.str.contains("giao hàng", regex=False).astype(int)
clean["Has_Curbside"] = service.str.contains("lề đường", regex=False).astype(int)
clean["Has_Drive_Through"] = service.str.contains("mua hàng ngay trên xe", regex=False).astype(int)

In [24]:
service_summary = clean[[
    "Has_Dine_In", "Has_Takeaway", "Has_Delivery",
    "Has_Curbside", "Has_Drive_Through"
]].mean().sort_values(ascending=False)

service_summary

Has_Dine_In          0.929846
Has_Delivery         0.609231
Has_Takeaway         0.563692
Has_Drive_Through    0.236923
Has_Curbside         0.061538
dtype: float64

Service_Options is useful because it can be converted into business features. These features help compare offline-first versus online-ready restaurants.

## 9. Digital-readiness features

In [25]:
digital_cols = ["Website", "Menu_Link", "Phone"]

clean[digital_cols].isna().mean().sort_values(ascending=False) * 100

Menu_Link    80.861538
Website      55.200000
Phone        14.953846
dtype: float64

In [26]:
clean["Has_Website"] = clean["Website"].notna().astype(int)
clean["Has_Menu"] = clean["Menu_Link"].notna().astype(int)
clean["Has_Phone"] = clean["Phone"].notna().astype(int)

clean["Digital_Readiness_Score"] = (
    clean["Has_Website"]
    + clean["Has_Menu"]
    + clean["Has_Phone"]
    + clean["Has_Delivery"]
)

clean[["Has_Website", "Has_Menu", "Has_Phone", "Has_Delivery", "Digital_Readiness_Score"]].head(10)

,Has_Website,Has_Menu,Has_Phone,Has_Delivery,Digital_Readiness_Score
0,0,0,1,1,2
1,1,0,1,0,2
2,1,1,1,1,4
3,0,0,1,1,2
4,1,0,1,1,3
5,1,0,1,1,3
6,0,0,1,1,2
7,1,1,1,1,4
8,0,0,1,1,2
9,0,1,1,1,3


## 10. Rating and review reliability

In [27]:
clean[["Avg_Rating", "Total_Reviews"]].describe()

,Avg_Rating,Total_Reviews
count,1622.000000,1622.000000
mean,4.419914,605.434649
std,0.365852,1281.619843
min,1.800000,1.000000
25%,4.100000,31.000000
50%,4.400000,196.500000
75%,4.700000,669.250000
max,5.000000,22395.000000


In [28]:
clean[
    (clean["Avg_Rating"] >= 4.9) &
    (clean["Total_Reviews"] < 20)
][["Name", "Area", "Avg_Rating", "Total_Reviews"]].head(20)

,Name,Area,Avg_Rating,Total_Reviews
25,Cafe Trà Sữa,Dĩ An,5.0,2.0
28,Quán ăn Dĩ An,Dĩ An,5.0,1.0
31,Hải sản Hải Bắc,Dĩ An,4.9,12.0
112,PHỞ GÀ NAM ĐỊNH,Dĩ An,5.0,2.0
116,ĂN VẶT NGỌC NGÂN | ĐỒ ĂN VẶT DĨ AN | TIỆM ĂN V...,Dĩ An,5.0,1.0
153,Quán ăn ngon tân phú,Quận 11,5.0,1.0
222,Súp Cua Nhà Thờ Đức Bà,Phú Nhuận,5.0,14.0
364,Nostalgia Tea & Coffee | COFFEE VIEW ĐẸP QUẬN ...,Quận 1,5.0,8.0
412,TÀU THỐ QUÁN| TÀU THỐ NGON QUẬN 3 | TÀU THỐ TR...,Quận 3,5.0,9.0
459,CAFE GIÓ,Quận 3,4.9,14.0


High rating with very low review count can be misleading. Therefore, raw Avg_Rating should not be used directly for restaurant ranking.

In [29]:
C = clean["Avg_Rating"].mean()
m = clean["Total_Reviews"].median()

clean["Bayesian_Score"] = (
    (clean["Total_Reviews"] / (clean["Total_Reviews"] + m)) * clean["Avg_Rating"]
    + (m / (clean["Total_Reviews"] + m)) * C
)

clean.sort_values("Bayesian_Score", ascending=False)[
    ["Name", "Area", "Avg_Rating", "Total_Reviews", "Bayesian_Score"]
].head(20)

,Name,Area,Avg_Rating,Total_Reviews,Bayesian_Score
908,Garden in Island by Mars Venus,Quận 7,5.0,5772.0,4.980902
493,Du thuyền Nhà hàng Benthanh Princess,Quận 4,5.0,3200.0,4.966440
200,LONG WANG CityLand Park Hills,Gò Vấp,5.0,1809.0,4.943163
1301,Hoàng's Kitchen,Quận 1,4.9,19063.0,4.895102
1292,Tung's Restaurant - Vietnamese Cuisine & veget...,Quận 1,4.9,10848.0,4.891458
944,Panda Tên Lửa,Bình Tân,4.9,7418.0,4.887611
228,Tiệm Nướng NỌ,Tân Bình,4.9,7148.0,4.887155
494,Uchi Sushi,Quận 4,4.9,6194.0,4.885238
1188,Panda Nguyễn Thị Thập,Quận 7,4.9,6077.0,4.884963
1293,Hai’s Restaurant,Quận 1,4.9,5686.0,4.883963


In [30]:
#create Opportunity_Score
review_log = np.log1p(clean["Total_Reviews"])

review_norm = (
    (review_log - review_log.min()) /
    (review_log.max() - review_log.min())
)

clean["Opportunity_Score"] = (
    review_norm * 0.4
    + (clean["Avg_Rating"] / 5) * 0.3
    + clean["Has_Delivery"] * 0.2
    + (1 - clean[["Has_Website", "Has_Menu"]].max(axis=1)) * 0.1
)

## 11. Star-rating consistency check

This is where the original notebook was failing because star columns were still strings.

In [31]:
star_cols = [
    "Rating_1_Stars", "Rating_2_Stars", "Rating_3_Stars",
    "Rating_4_Stars", "Rating_5_Stars"
]

clean["Star_Total"] = clean[star_cols].sum(axis=1, min_count=1)

clean["Review_Count_Diff"] = clean["Star_Total"] - clean["Total_Reviews"]

clean[[
    "Name", "Total_Reviews", "Star_Total", "Review_Count_Diff"
]].sort_values("Review_Count_Diff", key=abs, ascending=False).head(20)

,Name,Total_Reviews,Star_Total,Review_Count_Diff
288,Nhà Hàng Tiệc Cưới Bách Việt,1099.0,22401,21302.0
287,Trung Tâm Yến Tiệc Và Hội Nghị Aqua Jardin,1733.0,22401,20668.0
289,Trung Tâm Hội Nghị - Tiệc Cưới Riverside Palace,1939.0,22401,20462.0
286,Trung tâm Hội nghị - Tiệc cưới Diamond Place,3586.0,22401,18815.0
285,Nhà hàng tiệc cưới Grand Palace,5545.0,22401,16856.0
261,Du Miên Garden Cafe,11716.0,231,-11485.0
129,Làng Nướng Nam Bộ,9215.0,1186,-8029.0
231,Quán Hải Sản Ngon Tân Bình,31.0,7148,7117.0
229,309 ĐỒNG HƯƠNG QUÁN QUÁN NHẬU NGON RẺ TÂN BÌNH...,52.0,7148,7096.0
232,Đặc Khu Ăn Nhậu,336.0,7148,6812.0


In [32]:
clean["Bad_Review_Rate"] = np.where(
    clean["Star_Total"] > 0,
    (clean["Rating_1_Stars"] + clean["Rating_2_Stars"]) / clean["Star_Total"],
    np.nan
)

clean["Positive_Review_Rate"] = np.where(
    clean["Star_Total"] > 0,
    clean["Rating_5_Stars"] / clean["Star_Total"],
    np.nan
)

clean[["Name", "Bad_Review_Rate", "Positive_Review_Rate"]].head(10)

,Name,Bad_Review_Rate,Positive_Review_Rate
0,Nhà Hàng Ẩm Thực Bờ Sông,0.154717,0.433962
1,Subin BBQ Vincom Dĩ An Buffet Nướng Hàn Quốc,0.075922,0.817787
2,Làng Ẩm Thực 126 - Tiệc trọn gói hoàn hảo tại ...,0.131148,0.590164
3,Nhà hàng ẨM THỰC SÂN VƯỜN,0.277778,0.611111
4,Ẩm Thực Song Phụng 2 I Quán Ăn Sân Vườn Quận 1...,0.127660,0.723404
5,ẨM THỰC SÔNG XANH,0.086837,0.448812
6,Nhà Hàng Dìn Ký Phú Long,0.073394,0.599388
7,Nhà Hàng Vạn Lộc Phát,0.078740,0.803150
8,Năm Lửa 6,0.125000,0.430556
9,Béo BBQ,0.103787,0.712482


## 12. Duplicate check

In [33]:
duplicate_summary = pd.DataFrame({
    "check": [
        "Duplicated URL",
        "Duplicated Name",
        "Duplicated Name + Address"
    ],
    "duplicate_count": [
        clean.duplicated("URL").sum(),
        clean.duplicated("Name").sum(),
        clean.duplicated(["Name", "Address"]).sum()
    ]
})

duplicate_summary

,check,duplicate_count
0,Duplicated URL,0
1,Duplicated Name,16
2,Duplicated Name + Address,2


In [34]:
clean[clean.duplicated(["Name", "Address"], keep=False)][
    ["Name", "Address", "Avg_Rating", "Total_Reviews", "URL"]
].sort_values(["Name", "Address"]).head(30)

,Name,Address,Avg_Rating,Total_Reviews,URL
1339,Bếp Nhà Lục Tỉnh,"37 Nam Kỳ Khởi Nghĩa, Bến Thành, Quận 1, Hồ Ch...",4.0,1844.0,https://www.google.com/maps/place/B%E1%BA%BFp+...
1453,Bếp Nhà Lục Tỉnh,"37 Nam Kỳ Khởi Nghĩa, Bến Thành, Quận 1, Hồ Ch...",4.0,1844.0,https://www.google.com/maps/place/B%E1%BA%BFp+...
100,Hàng Dương Quán,"132 Đ. Số 65, Tân Phong, Quận 7, Hồ Chí Minh, ...",4.2,2873.0,https://www.google.com/maps/place/H%C3%A0ng+D%...
865,Hàng Dương Quán,"132 Đ. Số 65, Tân Phong, Quận 7, Hồ Chí Minh, ...",4.2,2873.0,https://www.google.com/maps/place/H%C3%A0ng+D%...


In [35]:
#Delete duplicate data
clean = clean.drop_duplicates(subset=["Name", "Address"], keep="first")

## 13. Final preview and export

In [36]:
clean[[
    "Name", "Address", "Province_City", "Area", "Market_Scope",
    "Avg_Rating", "Total_Reviews", "Bayesian_Score",
    "Price_Tier", "Has_Delivery", "Has_Website", "Has_Menu",
    "Digital_Readiness_Score", "Bad_Review_Rate", "Positive_Review_Rate"
]].head(20)

,Name,Address,Province_City,Area,Market_Scope,Avg_Rating,Total_Reviews,Bayesian_Score,Price_Tier,Has_Delivery,Has_Website,Has_Menu,Digital_Readiness_Score,Bad_Review_Rate,Positive_Review_Rate
0,Nhà Hàng Ẩm Thực Bờ Sông,"Đường Bờ Sông, Lái Thiêu, Thuận An, Bình Dương...",Bình Dương,Thuận An,Main Market,3.9,265.0,4.121372,NaN,1,0,0,2,0.154717,0.433962
1,Subin BBQ Vincom Dĩ An Buffet Nướng Hàn Quốc,"79 ĐT743B, Khu Phố Thống Nhất 1, Thành Phố, Bì...",Bình Dương,Dĩ An,Main Market,4.6,461.0,4.546180,200-300k,0,1,0,2,0.075922,0.817787
2,Làng Ẩm Thực 126 - Tiệc trọn gói hoàn hảo tại ...,"1/6 ĐT743B khu phố Bình Đức 1, Thuận An, Bình ...",Bình Dương,Thuận An,Main Market,4.1,61.0,4.344128,NaN,1,1,1,4,0.131148,0.590164
3,Nhà hàng ẨM THỰC SÂN VƯỜN,"16 Võ Thị Sáu, Đông Hòa, Dĩ An, Bình Dương, Vi...",Bình Dương,Dĩ An,Main Market,3.7,18.0,4.359501,300-400k,1,0,0,2,0.277778,0.611111
4,Ẩm Thực Song Phụng 2 I Quán Ăn Sân Vườn Quận 1...,"637/10/49 Tổ 30, Khu Phố 2, Quận 12, Hồ Chí Mi...",HCM,Quận 12,Main Market,4.3,47.0,4.396768,NaN,1,1,0,3,0.127660,0.723404
5,ẨM THỰC SÔNG XANH,"35 Khu phố Long thới Phường Lái Thiêu, Tp, Thu...",Bình Dương,Thuận An,Main Market,4.1,1094.0,4.148712,NaN,1,1,0,3,0.086837,0.448812
6,Nhà Hàng Dìn Ký Phú Long,"Số 31/4, Đường Đê bao sông Sài Gòn, Khu ph...",Bình Dương,Thuận An,Main Market,4.3,327.0,4.345011,NaN,1,0,0,2,0.073394,0.599388
7,Nhà Hàng Vạn Lộc Phát,"68-68A/12 QL13, Khu phố Trung, Thuận An, Bình ...",Bình Dương,Thuận An,Main Market,4.5,889.0,4.485503,NaN,1,1,1,4,0.078740,0.803150
8,Năm Lửa 6,"1 Đường N2, Khu Phố Thống Nhất 1, Dĩ An, Bình ...",Bình Dương,Dĩ An,Main Market,3.9,1008.0,3.984818,NaN,1,0,0,2,0.125000,0.430556
9,Béo BBQ,"45 Nguyễn Du, Dĩ An, Bình Dương, Việt Nam",Bình Dương,Dĩ An,Main Market,4.4,713.0,4.404302,NaN,1,0,1,3,0.103787,0.712482


In [37]:
final_missing = pd.DataFrame({
    "missing_count": clean.isna().sum(),
    "missing_rate_%": (clean.isna().mean() * 100).round(2)
}).sort_values("missing_rate_%", ascending=False)

final_missing.head(25)

,missing_count,missing_rate_%
Price_Range,1623,100.00
Highlights,1330,81.95
Menu_Link,1314,80.96
Website,897,55.27
Opening_Hours,896,55.21
Average_Price,805,49.60
Price_Tier,805,49.60
Phone,243,14.97
Bad_Review_Rate,135,8.32
Positive_Review_Rate,135,8.32


In [38]:
output_path = Path("../data/cleaned/restaurants_cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)


final_cols = [
    "_id", "Name", "Address",
    "Province_City", "Area", "Market_Scope",
    "Latitude", "Longitude",
    "Avg_Rating", "Total_Reviews",
    "Bayesian_Score", "Opportunity_Score",
    "Rating_1_Stars", "Rating_2_Stars", "Rating_3_Stars",
    "Rating_4_Stars", "Rating_5_Stars",
    "Star_Total", "Review_Count_Diff",
    "Bad_Review_Rate", "Positive_Review_Rate",
    "Average_Price", "Price_Tier",
    "Service_Options",
    "Has_Dine_In", "Has_Takeaway", "Has_Delivery",
    "Has_Curbside", "Has_Drive_Through",
    "Phone", "Website", "Menu_Link",
    "Has_Phone", "Has_Website", "Has_Menu",
    "Digital_Readiness_Score",
    "URL"
]

clean_final = clean[[c for c in final_cols if c in clean.columns]]

clean_final.to_csv(output_path, index=False, encoding="utf-8-sig")